In [1]:
from __future__ import annotations

import numpy as np
import sympy as sp
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader

from config.CEQLConfig import ModelTrainingConfig, NomtoConfig
from src.ComplexEQL import ComplexEQL
from src.utils import set_seed, train
from src.sympy_utils import filter_imaginary_part

# Dataset

In [2]:
def generate_xy_dataset(
    n_samples: int = 1000,
    x_range: tuple[float, float] = (-50.0, 50.0),   # x>0 to keep x^y real
    y_range: tuple[float, float] = (-50.0, 50.0),
):
    """
    Generate dataset for f(x, y) = x^y.
    We restrict x>0 so that x^y is real-valued for real y.
    Also filter out samples with |f| > 20 to avoid huge targets.
    """
    # sample x, y independently
    x = np.random.uniform(*x_range, size=(n_samples, 1))
    y = np.random.uniform(*y_range, size=(n_samples, 1))

    # compute function: f = x^y
    f = np.sin(x) # shape (N,1)

    # filter out non-finite values (just in case) and |f| > 20
    mask = np.isfinite(f) & (np.abs(f) <= 100.0)
    x = x[mask]
    y = y[mask]
    f = f[mask]

    # reshape again so concatenation never fails
    x = x.reshape(-1, 1)
    y = y.reshape(-1, 1)
    f = f.reshape(-1, 1)

    # concatenate safely: X = [x, y]
    X = np.concatenate([x, y], axis=1)

    return torch.from_numpy(X).float(), torch.from_numpy(f).float()

# Training

In [3]:
# -------------------------
# Config and setup
# -------------------------
mcfg = ModelTrainingConfig()
ncfg = NomtoConfig()

device = torch.device(mcfg.device)
set_seed(42)

# -------------------------
# Data
# -------------------------
X, y = generate_xy_dataset()
assert torch.isfinite(X).all(), "X contains NaN/Inf"
assert torch.isfinite(y).all(), "y contains NaN/Inf"
dataset = TensorDataset(X, y)
dataloader = DataLoader(
    dataset,
    batch_size=mcfg.train_batch_size,
    shuffle=True,
    drop_last=False,
)

# -------------------------
# Model, loss, optimizer, scheduler
# -------------------------
model = ComplexEQL(ncfg).to(device)
loss_fn = nn.MSELoss()

optimizer = torch.optim.Adam(model.parameters(), lr=mcfg.lr)
scheduler = None
if getattr(mcfg, "use_scheduler_phase3", False):
    if getattr(mcfg, "scheduler", None) == "ReduceLROnPlateau":
        scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
            optimizer, **mcfg.schedulerparams
        )

# -------------------------
# Training
# -------------------------
model, (sl0_weights, sl1_weights, al_weights, imaginary_out_losses, data_losses) = train(
    model=model,
    dataloader=dataloader,
    optimizer=optimizer,
    loss_fn=loss_fn,
    cfg=mcfg,
    device=device,
    scheduler=scheduler,
)

# -------------------------
# Symbolic readout
# -------------------------
x_sym, y_sym = sp.symbols("x y")
symbolic_expr = model.get_symbolic_expression([x_sym, y_sym], rounding_decimals=2)
print("\nDiscovered expression:")
print(symbolic_expr)

Random seed set as 42
[Phase 1 | Epoch 1] total=3.1826e+02, data=3.1826e+02, sparsity_reg=0.0000e+00, imag_w=0.0000e+00, alpha=1.000e-01, imag_coeff=1.000e-03, lr=1.00e-03
[Phase 1 | Epoch 100] total=1.0583e+02, data=1.0583e+02, sparsity_reg=0.0000e+00, imag_w=0.0000e+00, alpha=1.000e-01, imag_coeff=1.000e-03, lr=1.00e-03
[Phase 1 | Epoch 200] total=2.6153e+01, data=2.6153e+01, sparsity_reg=0.0000e+00, imag_w=0.0000e+00, alpha=1.000e-01, imag_coeff=1.000e-03, lr=1.00e-03
[Phase 1 | Epoch 300] total=2.6337e+00, data=2.6337e+00, sparsity_reg=0.0000e+00, imag_w=0.0000e+00, alpha=1.000e-01, imag_coeff=1.000e-03, lr=1.00e-03
[Phase 1 | Epoch 400] total=5.2973e-01, data=5.2973e-01, sparsity_reg=0.0000e+00, imag_w=0.0000e+00, alpha=1.000e-01, imag_coeff=1.000e-03, lr=1.00e-03
[Phase 1 | Epoch 500] total=5.0031e-01, data=5.0031e-01, sparsity_reg=0.0000e+00, imag_w=0.0000e+00, alpha=1.000e-01, imag_coeff=1.000e-03, lr=1.00e-03
[Phase 1 | Epoch 600] total=5.0007e-01, data=5.0007e-01, sparsity_re

KeyboardInterrupt: 

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# --------------------------------
# Grid along imaginary axis
# --------------------------------
t_min = 1e-6
t_max = 10.0
n = 4000
t = np.linspace(t_min, t_max, n)

gamma = 1.0
p = 1.0

# --------------------------------
# 1/x at Re(x)=0  →  x = i t
# --------------------------------
x_inv = 1j * t
inv = 1.0 / x_inv

# --------------------------------
# tan at Re(x)=pi/2  →  x = pi/2 + i t
# --------------------------------
x_tan = (np.pi / 2) + 1j * t
tanx = np.tan(x_tan)

# --------------------------------
# damped sin at Re(x)=pi/2
# Re(sin(u)) * exp(-gamma |t|^p), sin(pi/2)=1
# --------------------------------
damped_sin = np.exp(-gamma * (np.abs(t) ** p))

# --------------------------------
# damped tan
# --------------------------------
damped_tan = tanx * np.exp(-gamma * (np.abs(t) ** p))

# --------------------------------
# Prepare quantities
# --------------------------------
re_inv = np.abs(np.real(inv))
re_tan = np.abs(np.real(tanx))
re_damped_tan = np.abs(np.real(damped_tan))
re_damped_sin = np.abs(damped_sin)

im_inv = np.abs(np.imag(inv))
im_tan = np.abs(np.imag(tanx))
im_damped_tan = np.abs(np.imag(damped_tan))
im_damped_sin = np.zeros_like(t)

mag_inv = np.abs(inv)
mag_tan = np.abs(tanx)
mag_damped_tan = np.abs(damped_tan)
mag_damped_sin = np.abs(damped_sin)

# --------------------------------
# Plot
# --------------------------------
fig, axes = plt.subplots(1, 3, figsize=(18, 5), sharex=True)

# ---- Real parts
ax = axes[0]
ax.plot(t, re_inv, "-",  lw=2, label=r"$|\Re\{1/(it)\}|$")
ax.plot(t, re_tan, "--", lw=2, label=r"$|\Re\{\tan(\pi/2+it)\}|$")
ax.plot(t, re_damped_tan, "-.", lw=2, label=r"Damped $|\Re\{\tan\}|$")
ax.plot(t, re_damped_sin, ":", lw=2.5, label=r"Damped $|\Re\{\sin\}|$")
ax.set_yscale("log")
ax.set_title("Real part envelope")
ax.set_ylabel("Magnitude")
ax.grid(True, which="both", linewidth=0.4)
ax.legend()

# ---- Imag parts
ax = axes[1]
ax.plot(t, im_inv, "-",  lw=2, label=r"$|\Im\{1/(it)\}|$")
ax.plot(t, im_tan, "--", lw=2, label=r"$|\Im\{\tan(\pi/2+it)\}|$")
ax.plot(t, im_damped_tan, "-.", lw=2, label=r"Damped $|\Im\{\tan\}|$")
ax.plot(t, im_damped_sin, ":", lw=2.5, label=r"Damped $|\Im\{\sin\}|$")
ax.set_yscale("log")
ax.set_title("Imaginary part envelope")
ax.set_xlabel(r"$t = |\Im(x)|$")
ax.grid(True, which="both", linewidth=0.4)
ax.legend()

# ---- Magnitude
ax = axes[2]
ax.plot(t, mag_inv, "-",  lw=2, label=r"$|1/(it)|$")
ax.plot(t, mag_tan, "--", lw=2, label=r"$|\tan(\pi/2+it)|$")
ax.plot(t, mag_damped_tan, "-.", lw=2, label=r"Damped $|\tan|$")
ax.plot(t, mag_damped_sin, ":", lw=2.5, label=r"Damped $|\sin|$")
ax.set_yscale("log")
ax.set_title("Total magnitude")
ax.grid(True, which="both", linewidth=0.4)
ax.legend()

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# --------------------------------
# Grid in complex plane
# --------------------------------
u_min, u_max = -1.0, 1.0
v_min, v_max = -1.0, 1.0
n = 300

u = np.linspace(u_min, u_max, n)
v = np.linspace(v_min, v_max, n)
U, V = np.meshgrid(u, v)

# avoid division by zero
eps = 1e-6
U_safe = np.where(np.abs(U) < eps, np.sign(U) * eps + (U == 0) * eps, U)
V_safe = np.where(np.abs(V) < eps, np.sign(V) * eps + (V == 0) * eps, V)

Z = U_safe + 1j * V_safe
W = 1.0 / Z

ReW = np.real(W)
ImW = np.imag(W)
MagW = np.abs(W)

# --------------------------------
# Plot
# --------------------------------
fig = plt.figure(figsize=(18, 5))

# --- Re(1/x)
ax1 = fig.add_subplot(131, projection="3d")
ax1.plot_surface(U, V, ReW, rstride=4, cstride=4)
ax1.set_title("Re(1/x)")
ax1.set_xlabel("Re(x)")
ax1.set_ylabel("Im(x)")
ax1.set_zlabel("Value")

# --- Im(1/x)
ax2 = fig.add_subplot(132, projection="3d")
ax2.plot_surface(U, V, ImW, rstride=4, cstride=4)
ax2.set_title("Im(1/x)")
ax2.set_xlabel("Re(x)")
ax2.set_ylabel("Im(x)")
ax2.set_zlabel("Value")

# --- |1/x|
ax3 = fig.add_subplot(133, projection="3d")
ax3.plot_surface(U, V, MagW, rstride=4, cstride=4)
ax3.set_title("|1/x|")
ax3.set_xlabel("Re(x)")
ax3.set_ylabel("Im(x)")
ax3.set_zlabel("Magnitude")

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# --------------------------------
# Grid in complex plane
# --------------------------------
u_min, u_max = -np.pi, np.pi
v_min, v_max = -1.0, 1.0
n = 300

u = np.linspace(u_min, u_max, n)
v = np.linspace(v_min, v_max, n)
U, V = np.meshgrid(u, v)

Z = U + 1j * V
W = np.tan(Z)

ReW = np.real(W)
ImW = np.imag(W)
MagW = np.abs(W)

# --------------------------------
# Plot
# --------------------------------
fig = plt.figure(figsize=(18, 5))

# --- Re(tan(x))
ax1 = fig.add_subplot(131, projection="3d")
ax1.plot_surface(U, V, ReW, rstride=4, cstride=4)
ax1.set_title("Re(tan(x))")
ax1.set_xlabel("Re(x)")
ax1.set_ylabel("Im(x)")
ax1.set_zlabel("Value")

# --- Im(tan(x))
ax2 = fig.add_subplot(132, projection="3d")
ax2.plot_surface(U, V, ImW, rstride=4, cstride=4)
ax2.set_title("Im(tan(x))")
ax2.set_xlabel("Re(x)")
ax2.set_ylabel("Im(x)")
ax2.set_zlabel("Value")

# --- |tan(x)|
ax3 = fig.add_subplot(133, projection="3d")
ax3.plot_surface(U, V, MagW, rstride=4, cstride=4)
ax3.set_title("|tan(x)|")
ax3.set_xlabel("Re(x)")
ax3.set_ylabel("Im(x)")
ax3.set_zlabel("Magnitude")

plt.tight_layout()
plt.show()


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D

# --------------------------------
# Grid in complex plane
# --------------------------------
u_min, u_max = -10.0, 10.0
v_min, v_max = -10.0, 10.0
n = 300

u = np.linspace(u_min, u_max, n)
v = np.linspace(v_min, v_max, n)
U, V = np.meshgrid(u, v)

Z = U + 1j * V
W = np.sin(Z)

ReW = np.real(W)
ImW = np.imag(W)
MagW = np.abs(W)

# --------------------------------
# Plot
# --------------------------------
fig = plt.figure(figsize=(18, 5))

# --- Re(sin(x))
ax1 = fig.add_subplot(131, projection="3d")
ax1.plot_surface(U, V, ReW, rstride=4, cstride=4)
ax1.set_title("Re(sin(x))")
ax1.set_xlabel("Re(x)")
ax1.set_ylabel("Im(x)")
ax1.set_zlabel("Value")

# --- Im(sin(x))
ax2 = fig.add_subplot(132, projection="3d")
ax2.plot_surface(U, V, ImW, rstride=4, cstride=4)
ax2.set_title("Im(sin(x))")
ax2.set_xlabel("Re(x)")
ax2.set_ylabel("Im(x)")
ax2.set_zlabel("Value")

# --- |sin(x)|
ax3 = fig.add_subplot(133, projection="3d")
ax3.plot_surface(U, V, MagW, rstride=4, cstride=4)
ax3.set_title("|sin(x)|")
ax3.set_xlabel("Re(x)")
ax3.set_ylabel("Im(x)")
ax3.set_zlabel("Magnitude")

plt.tight_layout()
plt.show()


In [ ]:
x_sym, y_sym = sp.symbols("x y")
symbolic_expr = model.get_symbolic_expression([x_sym, y_sym], rounding_decimals=2)
# symbolic_expr = filter_imaginary_part(symbolic_expr, [x_sym, y_sym])
print(symbolic_expr)

In [ ]:
symbolic_expr

In [ ]:
sp.simplify(symbolic_expr)

# Visualize the predictions

In [ ]:
import numpy as np
import torch
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D  # OK to keep

# -------------------------------------------------------
# Grid over (x, y) for visualization
#   Match your data generation ranges by default
#   NOTE: your ground-truth uses log(x), so we must enforce x > 0
# -------------------------------------------------------
x_min, x_max = 1e-3, 50.0
y_min, y_max = -50.0, 50.0
grid_res = 200

xs = np.linspace(x_min, x_max, grid_res)
ys = np.linspace(y_min, y_max, grid_res)
Xg, Yg = np.meshgrid(xs, ys)

grid = np.stack([Xg, Yg], axis=-1)  # (R, R, 2)
grid_t = torch.from_numpy(grid.reshape(-1, 2)).float().to(device)

# -------------------------------------------------------
# Model predictions on grid
# -------------------------------------------------------
model.eval()
with torch.no_grad():
    pred = model(grid_t)  # (R*R, 1), complex
    pred_real = pred.real.detach().cpu().numpy().reshape(grid_res, grid_res)

# -------------------------------------------------------
# Ground truth: f(x,y) = 3 + 2.13 * log(x)
# -------------------------------------------------------
f_true = 3.0 + 2.13 * np.log(Xg)

# -------------------------------------------------------
# Shared limits (optional) to make surfaces comparable
# -------------------------------------------------------
z_min = np.min([pred_real.min(), f_true.min()])
z_max = np.max([pred_real.max(), f_true.max()])

# -------------------------------------------------------
# 3D plot: prediction surface
# -------------------------------------------------------
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")

ax.plot_surface(Xg, Yg, pred_real, cmap="viridis", linewidth=0, antialiased=True)

ax.set_title("Model Prediction Surface  f_pred(x,y) (real part)")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("f_pred")
ax.set_zlim(z_min, z_max)

plt.tight_layout()
plt.show()

# -------------------------------------------------------
# 3D plot: ground truth surface
# -------------------------------------------------------
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")

ax.plot_surface(Xg, Yg, f_true, cmap="plasma", linewidth=0, antialiased=True)

ax.set_title("Ground Truth Surface  f(x,y) = 3 + 2.13 log(x)")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("f_true")
ax.set_zlim(z_min, z_max)

plt.tight_layout()
plt.show()

# -------------------------------------------------------
# 3D plot: prediction error surface
# -------------------------------------------------------
fig = plt.figure(figsize=(10, 7))
ax = fig.add_subplot(111, projection="3d")

error = pred_real - f_true
ax.plot_surface(Xg, Yg, error, cmap="coolwarm", linewidth=0, antialiased=True)

ax.set_title("Prediction Error Surface  (model - truth)")
ax.set_xlabel("x")
ax.set_ylabel("y")
ax.set_zlabel("error")

plt.tight_layout()
plt.show()

# -------------------------------------------------------
# 2D diagnostic: slice at fixed y to verify y-invariance
# -------------------------------------------------------
y0 = 0.0
x_line = np.linspace(x_min, x_max, 600)
X_line = np.stack([x_line, np.full_like(x_line, y0)], axis=1)
X_line_t = torch.from_numpy(X_line).float().to(device)

model.eval()
with torch.no_grad():
    pred_line = model(X_line_t).real.detach().cpu().numpy().reshape(-1)

true_line = 3.0 + 2.13 * np.log(x_line)

plt.figure(figsize=(9, 4))
plt.plot(x_line, pred_line, label="pred (real)")
plt.plot(x_line, true_line, label="true", linestyle="--")
plt.xlabel("x")
plt.ylabel("f(x, y0)")
plt.title(f"Slice at y={y0}")
plt.legend()
plt.tight_layout()
plt.show()


# Visualize weights

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# ----------------------------
# Helper: flatten list of complex numpy arrays
# ----------------------------
def flatten_complex_list(weight_list):
    flat = [w.reshape(-1) for w in weight_list]
    mat = np.vstack(flat)
    return mat.real, mat.imag


# ==========================================================
# 1) Symbolic layer 0 weights
# ==========================================================
sl0_real, sl0_imag = flatten_complex_list(sl0_weights)
epochs0 = np.arange(len(sl0_real))

in_dim0, out_dim0 = sl0_weights[0].shape
labels_sl0 = [f"{i}{j}" for i in range(in_dim0) for j in range(out_dim0)]

# --- real part ---
plt.figure(figsize=(8, 5))
for k in range(sl0_real.shape[1]):
    plt.plot(epochs0, sl0_real[:, k], label=labels_sl0[k])
plt.title("Symbolic Layer 0 Weights (Real)")
plt.xlabel("Epoch")
plt.ylabel("Real part")
plt.grid(True)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize="small")
plt.tight_layout()

# --- imaginary part ---
plt.figure(figsize=(8, 5))
for k in range(sl0_imag.shape[1]):
    plt.plot(epochs0, sl0_imag[:, k], label=labels_sl0[k])
plt.title("Symbolic Layer 0 Weights (Imag)")
plt.xlabel("Epoch")
plt.ylabel("Imag part")
plt.grid(True)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize="small")
plt.tight_layout()


# ==========================================================
# 2) Assembly layer weights
# ==========================================================
al_real, al_imag = flatten_complex_list(al_weights)
epochs_al = np.arange(len(al_real))

labels_al = [f"{i:02d}" for i in range(al_real.shape[1])]

# --- real part ---
plt.figure(figsize=(8, 5))
for k in range(al_real.shape[1]):
    plt.plot(epochs_al, al_real[:, k], label=labels_al[k])
plt.title("Assembly Layer Weights (Real)")
plt.xlabel("Epoch")
plt.ylabel("Real part")
plt.grid(True)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize="small")
plt.tight_layout()

# --- imaginary part ---
plt.figure(figsize=(8, 5))
for k in range(al_imag.shape[1]):
    plt.plot(epochs_al, al_imag[:, k], label=labels_al[k])
plt.title("Assembly Layer Weights (Imag)")
plt.xlabel("Epoch")
plt.ylabel("Imag part")
plt.grid(True)
plt.legend(bbox_to_anchor=(1.05, 1), loc="upper left", fontsize="small")
plt.tight_layout()


# ==========================================================
# 3) Losses
# ==========================================================
imag_losses = np.array(imaginary_out_losses)
data_losses = np.array(data_losses)

# --- imaginary_out_losses ---
plt.figure(figsize=(6, 4))
plt.plot(imag_losses)
plt.title("Imaginary Output Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.grid(True)
# plt.yscale("log")
plt.tight_layout()

# --- data_losses (log scale) ---
plt.figure(figsize=(6, 4))
plt.plot(data_losses)
plt.title("Data Loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.yscale("log")    # <<<<<< LOG SCALE HERE
plt.grid(True, which="both")
plt.tight_layout()

plt.show()
